In [1]:
import numpy as np
import pickle
from sklearn.ensemble import ExtraTreesRegressor

from data.hiv_simulator import HIVSimulator

# Load Dataset

In [2]:
all_episodes = {}
all_episodes[0.5] = pickle.load(open('data/batch_trajectories_epsilon=05.p', 'rb'))
all_episodes[0.2] = pickle.load(open('data/batch_trajectories_epsilon=02.p', 'rb'))

In [3]:
print("Number of trajectories/episodes", len(all_episodes[0.2]))
print("Each episode comprises of {} lists. All lists are the same length. These represent (S, A, R, S', p) tuples".format(len(all_episodes[0.2][0])) )

Number of trajectories/episodes 250
Each episode comprises of 5 lists. All lists are the same length. These represent (S, A, R, S', p) tuples


In [4]:
# Examine the first episode
first_episode = all_episodes[0.2][0]

states = first_episode[0]
actions = first_episode[1]
rewards = first_episode[2]
next_states = first_episode[3]
propensities = first_episode[4] # probability of the selected action under the behavior policy

In [5]:
print( np.array(states[0:5]) )

[[5.21371162 0.69897    4.07718615 1.66275783 4.80562997 1.38021124]
 [5.30317939 1.74918268 2.93439581 1.43190928 3.53918668 1.4207034 ]
 [5.37917686 2.15822386 2.09293134 1.19502454 2.88451327 1.54718007]
 [5.43770298 2.05278582 3.06321797 1.87146423 3.79844829 1.65002342]
 [5.48880868 2.1022221  2.44742931 1.49258374 3.22973987 1.73588493]]


In [6]:
print( np.array(actions[0:5]) )

[3 1 0 1 3]


In [7]:
print( np.array(rewards[0:5]) )

[1.60192273 2.53750502 4.4042061  4.44661169 5.33127257]


In [8]:
print( np.array(next_states[0:5]) )

[[5.30317939 1.74918268 2.93439581 1.43190928 3.53918668 1.4207034 ]
 [5.37917686 2.15822386 2.09293134 1.19502454 2.88451327 1.54718007]
 [5.43770298 2.05278582 3.06321797 1.87146423 3.79844829 1.65002342]
 [5.48880868 2.1022221  2.44742931 1.49258374 3.22973987 1.73588493]
 [5.53366048 2.3834204  1.67247247 0.91825541 2.32646703 1.80149929]]


In [9]:
print( np.array(propensities[0:5]) )

[0.85 0.05 0.85 0.85 0.85]


# Offline RL

In this notebook, I will use several datasets of trajectories collected by various behavior policies. Specifically, the trajectories are collected from an RL environment simulator to model sequential decision-making in HIV treatment ([https://bitbucket.org/rlpy/rlpy/src/master/rlpy/Domains/HIVTreatment.py](https://bitbucket.org/rlpy/rlpy/src/master/rlpy/Domains/HIVTreatment.py)). I will evaluate the value of various policies using these datasets of trajectories.

**Overall information on environment.** The environment has a $6$-dimensional state space and $4$ actions. Rewards are not bounded. I will default to a horizon $H = 5$.

**Information on supporting datasets.**

- `data/batch_trajectories_epsilon=05.p`: Dataset of 250 i.i.d. trajectories collected using an $\epsilon$-greedy policy with $\epsilon=0.5$. Note this policy is fixed throughout the data collection period (is not learning). Includes state, action, reward, and action selection propensities.
- `data/batch_trajectories_epsilon=02.p`: Dataset of 250 i.i.d. trajectories collected using an $\epsilon$-greedy policy with $\epsilon=0.2$. Note this policy is fixed throughout the data collection period (is not learning). Includes state, action, reward, and action selection propensities.
- `data/fitted_Q_regressor.p`: Pickle file with a Q-function fitted using function approximation (using randomised decision trees).

In [10]:
H = 5 # horizon

I will start by using the dataset `data/batch_trajectories_epsilon=05.p` to evaluate the value of the uniform policy $\pi_u$, which selects each action with equal probability. In other words, I would like to estimate $\theta^* := \mathbb{E}_{\pi_u} \left[  \sum_{h=1}^H R_h \right]$.

We have that

$$
\theta^* = \mathbb{E}_{\pi_u} \left[  \sum_{h=1}^H R_h \right].
$$

Let $\pi_{\epsilon}$ denote the $\epsilon$-greedy policy. $\theta^*$ can then be written as

$$
\theta^* = \mathbb{E}_{\pi_{\epsilon}} \bigl[  G ( \tau_H ) W ( \tau_H; \pi_u, \pi_{\epsilon} ) \bigr],
$$

where

$$
G ( \tau_H ) = \sum_{h=1}^H R_h
$$

is the total reward and

$$
W ( \tau_H; \pi_u, \pi_{\epsilon} ) = \prod_{h=1}^H \frac { \pi_u ( A_h \mid S_h ) } { \pi_\epsilon ( A_h \mid S_h ) }
$$

is the importance sampling ratio between policies $\pi_u$ and $\pi_{\epsilon}$ of a given trajectory $\tau_H$.

The importance sampling estimator is then

$$
\hat{\theta^*} = \frac{1}{T} \sum_{t=1}^T G ( \tau_{H,t} ) \prod_{h=1}^H \frac { \pi_u ( A_{t,h} \mid S_{t,h} ) } { \pi_\epsilon ( A_{t,h} \mid S_{t,h} ) },
$$

where $T$ is the number of trajectories. By definition, $\pi_u ( A_{t,h} \mid S_{t,h} ) = 1/4$. $\pi_\epsilon ( A_{t,h} \mid S_{t,h} )$ is the action selection propensity provided in both datasets.

Using an asymptotic normal approximation,

$$
\text{95\% CI} = \left[ \hat{\theta^*} - 1.96 \sqrt{ \frac { \hat{\sigma}^2 } {T} }, \hat{\theta^*} + 1.96 \sqrt{ \frac { \hat{\sigma}^2 } {T} } \right],
$$

where $\hat{\sigma}^2$ is the sample variance.

In [11]:
# Load relevant episodes
episodes = all_episodes[0.5]
T = len(episodes) # number of episodes

# Initialise lists to store results
G = []  # list of cumulative rewards
W = []  # list of importance sampling ratios

# Compute G(τ_H) and W(τ_H; π_u, π_ϵ) for each episode
for episode in episodes:
    states, actions, rewards, next_states, propensities = episode
    cumulative_reward = sum(rewards[:H])
    importance_sampling_ratio = np.prod([(1/4) / p for p in propensities[:H]])
    G.append(cumulative_reward)
    W.append(importance_sampling_ratio)

# Compute importance sampling estimator
G = np.array(G)
W = np.array(W)
IS_estimate = np.mean(G * W)

# Confidence interval
std_error = np.sqrt(np.var(G * W) / T)

print(f'Importance Sampling Estimate for π_u (ϵ=0.5): {IS_estimate}')
print(f'95% Confidence Interval: [{IS_estimate - 1.96 * std_error}, {IS_estimate + 1.96 * std_error}]')

Importance Sampling Estimate for π_u (ϵ=0.5): 16.022712847503588
95% Confidence Interval: [10.240930904386314, 21.80449479062086]


I will now repeat the importance sampling estimation and confidence interval construction used above to estimate $\theta^* := \mathbb{E}_{\pi_u} \left[  \sum_{h=1}^H R_h \right]$ using the dataset `data/batch_trajectories_epsilon=02.p`.

In [12]:
# Load relevant episodes
episodes = all_episodes[0.2]
T = len(episodes) # number of episodes

# Initialise lists to store results
G = []  # list of cumulative rewards
W = []  # list of importance sampling ratios

# Compute G(τ_H) and W(τ_H; π_u, π_ϵ) for each episode
for episode in episodes:
    states, actions, rewards, next_states, propensities = episode
    cumulative_reward = sum(rewards[:H])
    importance_sampling_ratio = np.prod([(1/4) / p for p in propensities[:H]])
    G.append(cumulative_reward)
    W.append(importance_sampling_ratio)

# Compute importance sampling estimator
G = np.array(G)
W = np.array(W)
IS_estimate = np.mean(G * W)

# Confidence interval
std_error = np.sqrt(np.var(G * W) / T)

print(f'Importance Sampling Estimate for π_u (ϵ=0.2): {IS_estimate}')
print(f'95% Confidence Interval: [{IS_estimate - 1.96 * std_error}, {IS_estimate + 1.96 * std_error}]')

Importance Sampling Estimate for π_u (ϵ=0.2): 17.487166243967142
95% Confidence Interval: [-8.804967590971582, 43.77930007890586]


The estimates are very similar.

The $95\%$ confidence interval for $\epsilon = 0.5$ is relatively narrow, indicating more precision and less variability in the estimate. The $95\%$ confidence interval for $\epsilon = 0.2$ is relatively wide, indicating less precision and more variability in the estimate.

We start by discussing the exploration-exploitation trade-off. Intuitively, when $\epsilon = 0.5$, the policy explores more. This leads to a more diverse set of trajectories that covers more of the state-action space. When $\epsilon = 0.2$, the policy exploits more. This leads to a less diverse set of trajectories.

Mathematically, at each time step, an $\epsilon$-greedy policy selects: an action uniformly at random with probability $\epsilon$; and the action that maximises the expected total reward with probability $1 - \epsilon$.

We are now ready to discuss the key reason for the discrepancy: policy overlap. Intuitively, importance sampling relies on an overlap between the policy used to collect the trajectories, $\pi_{\epsilon}$, and the target policy, $\pi_u$. As $\epsilon$ decreases, $\pi_{\epsilon}$ becomes less exploratory, which decreases the overlap with the target policy, $\pi_u$.

Mathematically, recall the definition of the importance sampling ratio between policies $\pi_u$ and $\pi_{\epsilon}$ of a given trajectory $\tau_H$:

$$
W ( \tau_H; \pi_u, \pi_{\epsilon} ) = \prod_{h=1}^H \frac { \pi_u ( A_h \mid S_h ) } { \pi_\epsilon ( A_h \mid S_h ) }.
$$

As $\epsilon$ decreases, this term becomes more variable, which causes the wider confidence interval.

The formula for the effective sample size is

$$
\text{ESS} = \frac{ \left(\sum_{t=1}^T W_t \right)^2}{\sum_{t=1}^T W_t^2}
$$

where

$$
\quad W_t = \frac{\prod_{h'=1}^h \pi_u(A_{t,h'} \mid S_{t,h'})}{\prod_{h'=1}^h \pi_b(A_{t,h'} \mid S_{t,h'})}.
$$

The ESS can take values in the range $1 \leq \text{ESS} \leq T$.

Consider the case when the policy used to collect the trajectories and the target policy are perfectly aligned. We have that $W_1 = W_2 = \ldots = W_T = W$ and

$$
\text{ESS} = \frac{\left(\sum_{t=1}^T W \right)^2}{\sum_{t=1}^T W^2}
= \frac{\left(T W \right)^2}{T W^2}
= T.
$$

Now consider the case when the policy used to collect the trajectories and the target policy are poorly aligned. In the extreme case, one trajectory, $W_k$, dominates and

$$
\text{ESS} = \frac{\left(W_k \right)^2}{W_k^2}
= 1.
$$

We can therefore interpret the ESS as a metric that quantifies the overlap between the policy used to collect the trajectories and the target policy.

In [13]:
def compute_ess(W):
    """Compute the ESS"""
    W_sum = np.sum(W)
    W_squared_sum = np.sum(W**2)
    ess = (W_sum**2) / W_squared_sum
    return ess

In [14]:
# Compute ESS for ϵ=0.5

# Load relevant episodes
episodes = all_episodes[0.5]

# Initialise list to store results
W = []  # list of importance sampling ratios

for episode in episodes:
    states, actions, rewards, next_states, propensities = episode
    importance_sampling_ratio = np.prod([(1/4) / p for p in propensities[:H]])
    W.append(importance_sampling_ratio)

W = np.array(W)
ess = compute_ess(W)
print(f'Effective Sample Size for π_u (ϵ=0.5): {ess}')

Effective Sample Size for π_u (ϵ=0.5): 23.615677214314246


As discussed above, when $\epsilon = 0.5$, $\pi_{\epsilon}$ is more exploratory, which increases the overlap between the policy used to collect the trajectories, $\pi_{\epsilon}$, and the target policy, $\pi_u$. Therefore, the importance sampling ratios, $W_t$, are less variable and the ESS is larger.

In [15]:
# Compute ESS for ϵ=0.2

# Load relevant episodes
episodes = all_episodes[0.2]

# Initialise list to store results
W = []  # list of importance sampling ratios

for episode in episodes:
    states, actions, rewards, next_states, propensities = episode
    importance_sampling_ratio = np.prod([(1/4) / p for p in propensities[:H]])
    W.append(importance_sampling_ratio)

W = np.array(W)
ess = compute_ess(W)
print(f'Effective Sample Size for π_u (ϵ=0.2): {ess}')

Effective Sample Size for π_u (ϵ=0.2): 1.7810722282019509


Conversely, when $\epsilon = 0.2$, $\pi_{\epsilon}$ is less exploratory, which decreases the overlap between $\pi_{\epsilon}$ and $\pi_u$. Therefore, the importance sampling ratios, $W_t$, are more variable and the ESS is smaller.

Weighted importance sampling is an alternative estimation approach (in contrast to standard importance sampling). In weighted importance sampling, the importance sampling ratios are normalised. The importance sampling estimator is then

$$
\hat{\theta^*} = \frac { \sum_{t=1}^T G ( \tau_{H,t} ) \prod_{h=1}^H \frac { \pi_u ( A_{t,h} \mid S_{t,h} ) } { \pi_\epsilon ( A_{t,h} \mid S_{t,h} ) } } { \sum_{t=1}^T \prod_{h=1}^H \frac { \pi_u ( A_{t,h} \mid S_{t,h} ) } { \pi_\epsilon ( A_{t,h} \mid S_{t,h} ) } }.
$$

This reduces the variance of the estimator. This is useful when the policy used to collect the trajectories and the target policy have small overlap or when the sample size is small, as it reduces the disproportionate influence of some trajectories. However, weighted importance sampling introduces bias.

I will use weighted importance sampling to estimate $\mathbb{E}_{\pi_u} \left[ \sum_{h=1}^H R_h \right]$ using the dataset `data/batch_trajectories_epsilon=05.p`.

In [16]:
# Load relevant episodes
episodes = all_episodes[0.5]

# Initialise lists to store results
G = []  # list of cumulative rewards
W = []  # list of importance sampling ratios

# Compute G(τ_H) and W(τ_H; π_u, π_ϵ) for each episode
for episode in episodes:
    states, actions, rewards, next_states, propensities = episode
    cumulative_reward = sum(rewards[:H])
    importance_sampling_ratio = np.prod([(1/4) / p for p in propensities[:H]])
    G.append(cumulative_reward)
    W.append(importance_sampling_ratio)

# Compute weighted importance sampling estimator
G = np.array(G)
W = np.array(W)
WIS_estimate = np.sum(G * W) / np.sum(W)

print(f'Weighted Importance Sampling Estimate for π_u (ϵ=0.5): {WIS_estimate}')

Weighted Importance Sampling Estimate for π_u (ϵ=0.5): 13.198579960810633


The wide $95\%$ confidence interval obtained using standard importance sampling above suggests that the estimate could be skewed by outliers and weighted importance sampling could be more appropriate. Indeed, the weighted importance sampling estimate is within the $95\%$ confidence interval obtained using standard importance sampling, indicating that weighted importance sampling provides a reasonable estimate.

The Q-function `data/fitted_Q_regressor.p` was fit using Fitted Q-iteration on a separate dataset. I will now call the policy which always selects the best action according to this Q-function, $\pi_q$:

$$
\pi_q(s) = \text{argmax}_{a \in \mathcal{A}} Q(s, a).
$$

Note that this Q-function was fit using randomised decision trees as the regression function. Note also that it was fit using an infinite discounted horizon, so the Q-function is the same for all decision steps $h \in [1 \colon H]$. I will use importance sampling to estimate and construct a 95\% confidence interval for $\mathbb{E}_{\pi_q} \left[ \sum_{h=1}^H R_h \right]$ using the dataset `data/batch_trajectories_epsilon=02.p`.

In [ ]:
class FittedQ(object):
    def __init__(self, regressor=None):
        """Initialize simulator and regressor. Can optionally pass a custom
        `regressor` model (which must implement `fit` and `predict` -- you can
        use this to try different models like linear regression or NNs)"""
        self.simulator = HIVSimulator()
        self.regressor = regressor or ExtraTreesRegressor(n_estimators=10)
        self.action_codes = np.array(self.simulator.binary_action_codes)


    def Q(self, states):
        """Return the Q function estimate of `states` for each action"""
        # Uuse the trained regression model
        if len(states.shape) == 1:
            stateactions = np.concatenate( [np.tile(states,(4,1)), self.action_codes], axis=1 )
            rewards = self.regressor.predict( stateactions )
            return np.expand_dims(rewards,0)

        all_rewards = []
        for action in self.action_codes:
            stateactions = np.concatenate( [states, np.tile(action, (len(states), 1) )], axis=1 )
            rewards = self.regressor.predict( stateactions )
            all_rewards.append( rewards )
            
        return np.transpose( np.vstack( all_rewards ) )

regressor = pickle.load( open('data/fitted_Q_regressor.p', 'rb') )
Q_function = FittedQ(regressor)

In [18]:
def pi_q(state):
    """Return the selected action under π_q based on the Q-function"""
    q_values = Q_function.Q(state)[0]
    return np.argmax(q_values)

In [19]:
# Load relevant episodes
episodes = all_episodes[0.2]
T = len(episodes) # number of episodes

# Initialise lists to store results
G = []  # list of cumulative rewards
W = []  # list of importance sampling ratios

# Compute G(τ_H) and W(τ_H; π_u, π_ϵ) for each episode
for episode in episodes:
    states, actions, rewards, next_states, propensities = episode
    cumulative_reward = sum(rewards[:H])
    importance_sampling_ratio = 1.0
    for h in range(H):
        pi_q_propensity = 1.0 if actions[h] == pi_q(states[h]) else 0.0
        importance_sampling_ratio *= pi_q_propensity / propensities[h]
    G.append(cumulative_reward)
    W.append(importance_sampling_ratio)

# Compute importance sampling estimator
G = np.array(G)
W = np.array(W)
IS_estimate = np.mean(G * W)

# Confidence interval
std_error = np.sqrt(np.var(G * W) / T)

print(f'Importance Sampling Estimate for π_q (ϵ=0.2): {IS_estimate}')
print(f'95% Confidence Interval: [{IS_estimate - 1.96 * std_error}, {IS_estimate + 1.96 * std_error}]')

Importance Sampling Estimate for π_q (ϵ=0.2): 20.761611815698778
95% Confidence Interval: [18.03958063535283, 23.483642996044725]


The importance sampling estimate of the value of $\pi_q$ is higher than that of $\pi_u$ evaluated using the same dataset above. This is unsurprising. $\pi_q$ is designed to maximise the expected total reward. However, $\pi_u$ is entirely exploratory.

Furthermore, the $95\%$ confidence interval is significantly narrower for $\pi_q$. This suggests that, when $\epsilon = 0.2$, $\pi_{\epsilon}$ and $\pi_q$ have larger overlap than $\pi_{\epsilon}$ and $\pi_u$. This is unsurprising. Given that $\epsilon < 0.5$, we would expect $\pi_{\epsilon}$ to have larger overlap with $\pi_q$.